In [1]:
from langchain_community.llms.tongyi import Tongyi
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os
from langchain_community.embeddings import DashScopeEmbeddings
from langchain.graphs import Neo4jGraph
from knowledge_graph import make_kg
from knowledge_graph.kag import full_retriever
from langchain.embeddings import OllamaEmbeddings
from neo4j import GraphDatabase

load_dotenv()
os.environ["DASHSCOPE_API_KEY"] = "sk-b5883e47d69a417daae9f529e8e3ebf8"
url = "bolt://localhost:7687"
os.environ["NEO4J_URI"] = "bolt://localhost:7687"
os.environ["NEO4J_USERNAME"] = "neo4j"
os.environ["NEO4J_PASSWORD"] = "1609936983"
os.environ["NEO4J_DATABASE"] = "ckh"

llm1 = ChatOpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    model="qwen-max-latest",
    temperature=0,
    top_p=0.7
)
llm2 = Tongyi(model="qwen-max-latest",
              api_key="sk-b5883e47d69a417daae9f529e8e3ebf8",
              temperature=0,
              top_p=0.7)

embeddings = OllamaEmbeddings(model="bge-large")

graph_db = Neo4jGraph(
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
    database=os.environ["NEO4J_DATABASE"]
)

C:\Users\CKH-Pro16\AppData\Local\Temp\ipykernel_24600\2118120088.py:35: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="bge-large")
C:\Users\CKH-Pro16\AppData\Local\Temp\ipykernel_24600\2118120088.py:37: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-neo4j package and should be used instead. To use it run `pip install -U :class:`~langchain-neo4j` and import as `from :class:`~langchain_neo4j import Neo4jGraph``.
  graph_db = Neo4jGraph(


In [18]:
doc = make_kg.read_doc_for_kg(file_path="test_file/text.txt")

graph_documents = make_kg.make_kg(llm=llm2, documents=doc)

graph_db.add_graph_documents(
    graph_documents,
    baseEntityLabel=True,
    include_source=True
)


文档的分块数量为：9
共创建了9个知识图谱


In [20]:
graph_db.add_graph_documents(
    graph_documents,
    baseEntityLabel=True,
    include_source=True
)

In [19]:
def clear_database(tx):
    # 删除所有关系
    tx.run("MATCH ()-[r]->() DELETE r")
    # 删除所有节点
    tx.run("MATCH (n) DELETE n")


driver = GraphDatabase.driver(uri=os.environ["NEO4J_URI"],
                              database=os.environ["NEO4J_DATABASE"],
                              auth=(os.environ["NEO4J_USERNAME"],
                                    os.environ["NEO4J_PASSWORD"]))
with driver.session() as session:
    session.write_transaction(clear_database)
# 关闭驱动
driver.close()

C:\Users\CKH-Pro16\AppData\Local\Temp\ipykernel_14196\2077082468.py:12: DeprecationWarning: write_transaction has been renamed to execute_write
  session.write_transaction(clear_database)


In [2]:
print(full_retriever("就是大批的青年人“躺平”或者“啃老”", llm2, embeddings, graph_db))

['青年人', '躺平', '啃老']
['青年人', '躺平', '啃老']
Generated Query: 青年人
Generated Query: 躺平
Generated Query: 啃老
Graph data:
青年人 - ENGAGE_IN -> 啃老
青年人 - ENGAGE_IN -> 躺平
老年人 - NOT_ONLY_BURDEN_BUT_ALSO -> 能发挥很多积极能量的群体
中国老年人 - HAS_CHANGED -> 三个显著的变化
老人 - BELONG_TO -> 先富起来的老人
高净值人群 - DEFINED_BY -> 可投资资产在1000万以上
中国60岁以上的高净值人群 - HAS_PERCENTAGE -> 5%
啃老 - DEPENDS_ON -> 老人
老人 - BELONG_TO -> 先富起来的老人青年人 - ENGAGE_IN -> 躺平啃老 - DEPENDS_ON -> 老人
老人 - BELONG_TO -> 先富起来的老人
老年人 - NOT_ONLY_BURDEN_BUT_ALSO -> 能发挥很多积极能量的群体
中国老年人 - HAS_CHANGED -> 三个显著的变化
青年人 - ENGAGE_IN -> 啃老
啃老 - DEPENDS_ON -> 老人
老人 - BELONG_TO -> 先富起来的老人
vector data:

text: 当今社会不是有一种被诟病的现象吗？就是大批的青年人“躺平”或者“啃老”，“啃老”的前提条件就是有一大批可啃的老人，这些老人就是先富起来的老人。#Document 
text: 这是大面上的，还需要注意，这群老年人还有很多是富人。按照今年的统计，中国60岁以上的高净值人群，占这个群体的5%，已经很高了。什么是高净值人群呢？就是那些可投资资产在1000万以#Document 
text: 如果和过去中国几千年的传统社会做纵向比较，你就会发现，今天的中国老年人已经有了三个显著的变化，这些变化都指向一个方向，就是老年人不仅仅是负担，而是能发挥很多积极能量的群体。这三个#Document 
text: 第一，老年人不再只是一穷二白，守着两亩薄田过活的人。
    
Graph data:
青年人 - ENGAGE_IN -> 啃老
青年人 - ENGAGE_IN 

In [5]:
from langchain_core.prompts import PromptTemplate

template = """
仅根据下列上下文回答问题:
{context}

Question: {question}
使用自然语言，简洁明了.
"""

# 创建 PromptTemplate 对象
prompt_template = PromptTemplate(
    input_variables=["question", "context"],  # 模板中需要填充的变量
    template=template
)

question = "老人的钱来自哪里"
context = full_retriever(question, llm2, embeddings, graph_db)
print(prompt_template.format(question=question, context=context))
response = llm2.invoke(prompt_template.format(question=question, context=context))
print(response)

['老人', '钱']
['老人', '钱']
Generated Query: 老人
Generated Query: 钱
Graph data:
老人 - BELONG_TO -> 先富起来的老人
老年人 - NOT_ONLY_BURDEN_BUT_ALSO -> 能发挥很多积极能量的群体
中国老年人 - HAS_CHANGED -> 三个显著的变化
啃老 - DEPENDS_ON -> 老人
青年人 - ENGAGE_IN -> 啃老
青年人 - ENGAGE_IN -> 躺平
高净值人群 - DEFINED_BY -> 可投资资产在1000万以上
中国60岁以上的高净值人群 - HAS_PERCENTAGE -> 5%
啃老 - DEPENDS_ON -> 老人
老人 - BELONG_TO -> 先富起来的老人
青年人 - ENGAGE_IN -> 啃老
vector data:

text: 这是大面上的，还需要注意，这群老年人还有很多是富人。按照今年的统计，中国60岁以上的高净值人群，占这个群体的5%，已经很高了。什么是高净值人群呢？就是那些可投资资产在1000万以#Document 
text: 和过去中国历史任何一个时期相比，我们这个时代的老人肯定更有钱，这是不言而喻的。这个钱可能是房子，也可能是退休金，甚至有可能是家里面农村的宅基地。即使没有这些，也有国家兜底的养老金#Document 
text: 当今社会不是有一种被诟病的现象吗？就是大批的青年人“躺平”或者“啃老”，“啃老”的前提条件就是有一大批可啃的老人，这些老人就是先富起来的老人。#Document 
text: 第一，老年人不再只是一穷二白，守着两亩薄田过活的人。
    

仅根据下列上下文回答问题:
Graph data:
老人 - BELONG_TO -> 先富起来的老人
老年人 - NOT_ONLY_BURDEN_BUT_ALSO -> 能发挥很多积极能量的群体
中国老年人 - HAS_CHANGED -> 三个显著的变化
啃老 - DEPENDS_ON -> 老人
青年人 - ENGAGE_IN -> 啃老
青年人 - ENGAGE_IN -> 躺平
高净值人群 - DEFINED_BY -> 可投资资产在1000万以上
中国60岁以上的高净值人群 - HAS_PERCENT